## Step 1: Install Libraries & Mount Drive

In [1]:
# Install necessary libraries (including the fix for audio errors)
!pip install transformers datasets librosa jiwer accelerate torchaudio
!pip install torchcodec soundfile

from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import glob
from datasets import Dataset, Audio, ClassLabel
import torch
import librosa
import json

# Verify GPU is available
print(f"GPU Available: {torch.cuda.is_available()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU Available: True


## Step 2: Data Preparation

In [2]:
# Configuration
BASE_PATH = "/content/drive/MyDrive/shobdotori"
TRAIN_AUDIO_DIR = os.path.join(BASE_PATH, "Train")
ANNOTATION_DIR = os.path.join(BASE_PATH, "Train_annotation")

# 1. Load Data Metadata
data_records = []
region_labels = set()
csv_files = glob.glob(os.path.join(ANNOTATION_DIR, "*.csv"))

print(f"Found {len(csv_files)} annotation files.")

for csv_file in csv_files:
    region_name = os.path.basename(csv_file).replace('.csv', '')
    region_labels.add(region_name)
    df = pd.read_csv(csv_file)

    for idx, row in df.iterrows():
        audio_path = os.path.join(TRAIN_AUDIO_DIR, region_name, row['audio'])
        if os.path.exists(audio_path):
            data_records.append({
                "audio": audio_path,
                "sentence": row['text'],
                "label": region_name
            })

# 2. Create Dataset
df_dataset = pd.DataFrame(data_records)
region_list = sorted(list(region_labels))
class_label = ClassLabel(num_classes=len(region_list), names=region_list)
dataset = Dataset.from_pandas(df_dataset)

# 3. CRITICAL UPDATE: Filter BOTH Long and Short Audio
print(f"Original size: {len(dataset)}")

def filter_audio_length(example):
    try:
        duration = librosa.get_duration(path=example["audio"])
        # FIX: Keep files between 1 second and 12 seconds
        # > 1.0s prevents "mask_length" error
        # < 12.0s prevents "Out of Memory" error
        return duration > 1.0 and duration < 12.0
    except:
        return False

dataset = dataset.filter(filter_audio_length)
print(f"Filtered size: {len(dataset)} (Safe for Training)")

# 4. Cast Audio
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

Found 20 annotation files.
Original size: 3350


Filter:   0%|          | 0/3350 [00:00<?, ? examples/s]

Filtered size: 3321 (Safe for Training)


## Step 3: Train ASR Model (Text Detection)

In [3]:
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor, TrainingArguments, Trainer, TrainerCallback
from transformers import Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Union
import shutil
import os
import json
import torch

# --- 1. SETUP CALLBACK FOR GOOGLE DRIVE BACKUP ---
# This saves your work to Drive automatically. If Colab crashes, you lose nothing.
class SaveToDriveCallback(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        # Define where in Drive to save
        drive_checkpoint_dir = os.path.join("/content/drive/MyDrive/shobdotori/checkpoints", f"checkpoint-{state.global_step}")

        # Copy the local checkpoint to Google Drive
        if os.path.exists(args.output_dir):
            try:
                # Remove if exists (to overwrite)
                if os.path.exists(drive_checkpoint_dir):
                    shutil.rmtree(drive_checkpoint_dir)

                # Copy current checkpoint folder to Drive
                last_checkpoint = os.path.join(args.output_dir, f"checkpoint-{state.global_step}")
                if os.path.exists(last_checkpoint):
                    shutil.copytree(last_checkpoint, drive_checkpoint_dir)
                    print(f"\n✅ SECURE BACKUP: Saved checkpoint-{state.global_step} to Google Drive!")
            except Exception as e:
                print(f"\n⚠️ Warning: Could not backup to Drive: {e}")

# --- 2. VOCAB & PROCESSOR ---
def extract_all_chars(batch):
    all_text = " ".join(batch["sentence"])
    vocab = list(set(all_text))
    return {"vocab": [vocab], "all_text": [all_text]}

vocabs = dataset.map(extract_all_chars, batched=True, batch_size=-1, keep_in_memory=True, remove_columns=dataset.column_names)
vocab_list = list(set(vocabs["vocab"][0]))
vocab_dict = {v: k for k, v in enumerate(vocab_list)}
vocab_dict["|"] = vocab_dict[" "]
del vocab_dict[" "]
vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)

with open('vocab.json', 'w') as vocab_file:
    json.dump(vocab_dict, vocab_file)

tokenizer = Wav2Vec2CTCTokenizer("./vocab.json", unk_token="[UNK]", pad_token="[PAD]", word_delimiter_token="|")
feature_extractor = Wav2Vec2FeatureExtractor(feature_size=1, sampling_rate=16000, padding_value=0.0, do_normalize=True, return_attention_mask=True)
processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)

# --- 3. PREPARE DATA ---
def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_values"] = processor(audio["array"], sampling_rate=audio["sampling_rate"]).input_values[0]
    with processor.as_target_processor():
        batch["labels"] = processor(batch["sentence"]).input_ids
    return batch

encoded_dataset = dataset.map(prepare_dataset, remove_columns=dataset.column_names)

# --- 4. DATA COLLATOR ---
@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_values": feature["input_values"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        with self.processor.as_target_processor():
            labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor)

# --- 5. MODEL & TRAINING ---
model = Wav2Vec2ForCTC.from_pretrained(
    "facebook/wav2vec2-large-xlsr-53",
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer)
)

training_args = TrainingArguments(
    output_dir="./wav2vec2-bangla-asr",
    group_by_length=True,
    per_device_train_batch_size=1,   # Safe for T4
    gradient_accumulation_steps=8,   # Safe for T4
    gradient_checkpointing=True,     # Memory Saver
    num_train_epochs=10,
    fp16=True,
    save_steps=500,                  # Backup every 500 steps
    learning_rate=3e-4,
    warmup_steps=500,
    save_total_limit=1,              # Keep 1 local copy
    logging_steps=50,
    dataloader_num_workers=0
)

trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    train_dataset=encoded_dataset,
    tokenizer=processor.feature_extractor,
    callbacks=[SaveToDriveCallback]  # <--- THIS ACTIVATES THE DRIVE BACKUP
)

trainer.train()

# --- 6. FINAL SAVE TO DRIVE ---
trainer.save_model("./final_asr_model")
processor.save_pretrained("./final_asr_model")

# Force copy the final model to Drive
final_drive_path = "/content/drive/MyDrive/shobdotori/final_asr_model"
if os.path.exists(final_drive_path):
    shutil.rmtree(final_drive_path)
shutil.copytree("./final_asr_model", final_drive_path)
print(f"✅ Training Complete. Final model saved to: {final_drive_path}")

Map:   0%|          | 0/3321 [00:00<?, ? examples/s]

Map:   0%|          | 0/3321 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:180: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-large-xlsr-53 and

 3


wandb: You chose "Don't visualize my results"


/usr/local/lib/python3.12/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:180: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/backends/cudnn/__init__.py:145: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  torch._C._get_cudn

Step,Training Loss
50,24.459200
100,17.252300
150,4.940300
200,3.642700
250,3.593500
300,3.506300
350,3.407700
400,3.157900
450,3.208300
500,3.016800


/usr/local/lib/python3.12/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:180: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:180: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:180: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call

[]

## Step 4: Train Region Classifier (Dialect Detection)

In [4]:
from transformers import AutoModelForAudioClassification, TrainingArguments, Trainer
from transformers import Wav2Vec2FeatureExtractor
from datasets import Dataset, Audio
import shutil
import os

# 1. Feature Extractor
feature_extractor = Wav2Vec2FeatureExtractor(feature_size=1, sampling_rate=16000, padding_value=0.0, do_normalize=True, return_attention_mask=True)

# 2. Prepare Data
def prepare_classification(batch):
    audio = batch["audio"]
    batch["input_values"] = feature_extractor(
        audio["array"],
        sampling_rate=16000,
        max_length=16000*5, # 5 seconds
        truncation=True,
        padding="max_length"
    ).input_values[0]

    # --- BUG FIX HERE ---
    # Old Code: batch["label"] (Singular) -> CAUSED CRASH
    # New Code: batch["labels"] (Plural) -> REQUIRED BY TRAINER
    batch["labels"] = class_label.str2int(batch["label"])
    return batch

# Use 'dataset' from Cell 2 (Pre-filtered)
dataset_cls = dataset
encoded_dataset_cls = dataset_cls.map(prepare_classification, remove_columns=["audio", "sentence", "label"])

# 3. Model
num_labels = len(region_list)
model_cls = AutoModelForAudioClassification.from_pretrained(
    "facebook/wav2vec2-base",
    num_labels=num_labels,
    label2id={l: i for i, l in enumerate(region_list)},
    id2label={i: l for i, l in enumerate(region_list)}
)

# 4. Train
training_args_cls = TrainingArguments(
    output_dir="./wav2vec2-bangla-region-cls",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=1,
    num_train_epochs=5,
    learning_rate=3e-5,
    fp16=True,
    save_strategy="epoch",
    logging_steps=50,
)

trainer_cls = Trainer(
    model=model_cls,
    args=training_args_cls,
    train_dataset=encoded_dataset_cls,
    tokenizer=feature_extractor,
)

trainer_cls.train()

# 5. Save to Drive
trainer_cls.save_model("./final_region_model")
final_region_path = "/content/drive/MyDrive/shobdotori/final_region_model"
if os.path.exists(final_region_path):
    shutil.rmtree(final_region_path)
shutil.copytree("./final_region_model", final_region_path)
print(f"✅ Region Model saved to: {final_region_path}")

Map:   0%|          | 0/3321 [00:00<?, ? examples/s]

config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/configuration_utils.py:335: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/380M [00:00<?, ?B/s]

Some weights of Wav2Vec2ForSequenceClassification were not initialized from the model checkpoint at facebook/wav2vec2-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'projector.bias', 'projector.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-1860251670.py:43: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_cls = Trainer(


model.safetensors:   0%|          | 0.00/380M [00:00<?, ?B/s]

ValueError: The model did not return a loss from the inputs, only the following keys: logits. For reference, the inputs it received are input_values,attention_mask.

## Step 5: Inference on Test Audio

In [ ]:
import torch
import pandas as pd
import glob
import os
import librosa
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor, AutoModelForAudioClassification

# 1. Setup Paths
#    Make sure this matches your folder structure
TEST_DIR = os.path.join(BASE_PATH, "Test")
test_files = glob.glob(os.path.join(TEST_DIR, "*.wav"))
print(f"Found {len(test_files)} test files.")

results = []

# 2. Load Processor & Models
#    We load the processor from the saved ASR model directory to ensure valid vocab
print("Loading models...")
processor = Wav2Vec2Processor.from_pretrained("./final_asr_model")

#    Load models to GPU (cuda)
asr_model = Wav2Vec2ForCTC.from_pretrained("./final_asr_model").to("cuda")
cls_model = AutoModelForAudioClassification.from_pretrained("./final_region_model").to("cuda")

print("Starting Inference...")

# 3. Inference Loop
for audio_file in test_files:
    try:
        # Load Audio (Resample to 16kHz)
        speech, rate = librosa.load(audio_file, sr=16000)

        # --- TASK A: TEXT RECOGNITION (ASR) ---
        # ASR needs the FULL audio
        inputs_asr = processor(speech, sampling_rate=16000, return_tensors="pt", padding=True)
        with torch.no_grad():
            logits_asr = asr_model(inputs_asr.input_values.to("cuda")).logits

        pred_ids = torch.argmax(logits_asr, dim=-1)
        transcription = processor.batch_decode(pred_ids)[0]

        # --- TASK B: REGION DETECTION (CLS) ---
        # CLS was trained on 5s chunks. We should truncate to match training.
        # 5 seconds * 16000 Hz = 80000 samples
        max_cls_len = 80000
        speech_cls = speech[:max_cls_len] if len(speech) > max_cls_len else speech

        inputs_cls = processor.feature_extractor(
            speech_cls,
            sampling_rate=16000,
            return_tensors="pt",
            padding="max_length",
            max_length=max_cls_len,
            truncation=True
        )

        with torch.no_grad():
            logits_cls = cls_model(inputs_cls.input_values.to("cuda")).logits

        pred_region_id = torch.argmax(logits_cls, dim=-1).item()
        # Use cls_model.config (safe) instead of model_cls (unsafe)
        region_name = cls_model.config.id2label[pred_region_id]

        # Store Result
        results.append({
            "filename": os.path.basename(audio_file),
            "predicted_text": transcription,
            "predicted_region": region_name
        })

    except Exception as e:
        print(f"Error processing {os.path.basename(audio_file)}: {e}")
        # Add placeholder to keep CSV valid
        results.append({
            "filename": os.path.basename(audio_file),
            "predicted_text": "",
            "predicted_region": "Error"
        })

# 4. Save Submission
df_submission = pd.DataFrame(results)
df_submission.to_csv("submission.csv", index=False)
print("✅ Inference Complete! Saved to submission.csv")
print(df_submission.head())